# ⚡ PySpark & HDFS Distributed Data Engineering Pipeline

**Apache Hadoop & Spark Unified Ecosystem**  
This interactive notebook demonstrates end-to-end distributed data processing with Apache Spark and Hadoop Distributed File System (HDFS):
1. Connecting to the distributed HDFS RPC layer (`hdfs://localhost:9000` or `hdfs://hadoop:9000`).
2. Ingesting raw CSV datasets from HDFS into Apache Spark DataFrames.
3. Running high-performance transformations, filtering, and aggregations.
4. Persisting results back to HDFS as partitioned Snappy-compressed **Apache Parquet** files.
5. Reading back and benchmarking Parquet column projection and predicate pushdown.

In [ ]:
# Step 1: Initialize PySpark Session with HDFS Configuration
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round, when

spark = SparkSession.builder \
    .appName("PySpark-HDFS-DataPipeline") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"⚡ SparkSession Active: Spark Version {spark.version}")
print(f"🐘 Target FileSystem: {spark.conf.get('spark.hadoop.fs.defaultFS')}")

In [ ]:
# Step 2: Ingest Sample Dataset or Create Distributed DataFrame
data = [
    (1, "Alice", "Engineering", 95000.0, "Cairo", "2026-01-15"),
    (2, "Bob", "Marketing", 68000.0, "Alexandria", "2026-02-01"),
    (3, "Charlie", "Engineering", 105000.0, "Giza", "2026-02-15"),
    (4, "Diana", "Data Science", 112000.0, "Cairo", "2026-03-01"),
    (5, "Evan", "Marketing", 72000.0, "Cairo", "2026-03-10"),
    (6, "Fiona", "Data Science", 120000.0, "Alexandria", "2026-03-20"),
    (7, "George", "DevOps", 98000.0, "Giza", "2026-04-05"),
    (8, "Hana", "Engineering", 102000.0, "Cairo", "2026-04-12")
]

columns = ["EmployeeID", "Name", "Department", "Salary", "City", "HireDate"]
df_raw = spark.createDataFrame(data, columns)

print("=== Source DataFrame Schema ===")
df_raw.printSchema()
df_raw.show()

In [ ]:
# Step 3: Analytical Transformations & Aggregations
df_transformed = df_raw.withColumn(
    "SalaryTier",
    when(col("Salary") >= 100000, "Senior / Lead").otherwise("Standard")
)

df_summary = df_transformed.groupBy("Department").agg(
    count("EmployeeID").alias("Headcount"),
    round(avg("Salary"), 2).alias("AvgSalary")
).orderBy(col("AvgSalary").desc())

print("=== Department Salary Aggregations ===")
df_summary.show()

In [ ]:
# Step 4: Write Partitioned Parquet to Distributed HDFS Storage
hdfs_parquet_path = "hdfs://localhost:9000/data/spark_employees.parquet"
print(f"Writing Parquet dataset to: {hdfs_parquet_path}...")

df_transformed.write \
    .mode("overwrite") \
    .partitionBy("Department") \
    .parquet(hdfs_parquet_path)

print("🎉 Successfully written partitioned Parquet to HDFS!")

In [ ]:
# Step 5: Read Parquet Back From HDFS & Verify Predicate Pushdown
df_loaded = spark.read.parquet(hdfs_parquet_path)
print("Loaded Parquet Schema:")
df_loaded.printSchema()

# Filter on partitioned column (Department = 'Data Science')
ds_team = df_loaded.filter(col("Department") == "Data Science")
print("=== Data Science Team (Partition Filtered) ===")
ds_team.show()

# Stop SparkSession
spark.stop()
print("=== Pipeline Run Finished Successfully ===")